# VAE

In [ ]:
import sys
sys.path.append('/home/jovyan/work/Similarity-Aware-Label-Smoothing')

In [8]:
# Import required packages
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import models
import numpy as np
from tqdm import tqdm
from dataset_utils import get_data_loaders

## Hyperparams

In [9]:
dataset = "cifar100"
batch_size = 256
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
latent_dim = 128
lr = 2e-4
weight_decay = 1e-5
num_epochs = 200
kl_weight = 4


In [ ]:
print(dataset, latent_dim, kl_weight)

## VAE Structure

In [ ]:
class ConvVAE(nn.Module):
    def __init__(self, img_size=32, channel_num = 3, kernel_num = 128, layers_num = 3, latent_dim=256):
        super().__init__()
        self.img_size = img_size
        self.channel_num = channel_num
        self.kernel_num = kernel_num
        self.depth = layers_num
        self.latent_dim = latent_dim

        # ---------- Encoder ----------
        self.enc = nn.Sequential(
            self._conv(channel_num, kernel_num // (2 ** (layers_num - 1))),
            *[self._conv(kernel_num // (2 ** k), kernel_num // (2 ** (k - 1))) for k in range(layers_num - 1, 0, -1)],
        )

        self.feat_size = img_size // (2 ** layers_num)
        self.feat_dim = self.feat_size * self.feat_size * self.kernel_num

        self.mu = nn.Linear(self.feat_dim, latent_dim)
        self.logvar = nn.Linear(self.feat_dim, latent_dim)

        # ---------- Decoder ----------
        self.fc_dec = nn.Linear(latent_dim, self.feat_dim)

        self.dec = nn.Sequential(
            *[self._deconv(kernel_num // (2 ** k), kernel_num // (2 ** (k+1))) for k in range(layers_num - 1)],
            nn.ConvTranspose2d(kernel_num // (2 ** (layers_num - 1)), channel_num, 4, 2, 1),
        )

    def encode(self, x):
        h = self.enc(x)
        h = h.view(h.size(0), -1)
        mu = self.mu(h)
        logvar = self.logvar(h)
        return mu, logvar

    def reparameterize(self, mu, logvar):
        std = (0.5 * logvar).exp()
        eps = torch.randn_like(std)
        return mu + eps * std

    def decode(self, z):
        h = self.fc_dec(z)
        h = h.view(-1, self.kernel_num, self.feat_size, self.feat_size)
        return self.dec(h)

    def forward(self, x):
        mu, logvar = self.encode(x)
        z = self.reparameterize(mu, logvar)
        x_rec = self.decode(z)
        return x_rec, mu, logvar, z

    # Layers
    def _conv(self, channel_num, kernel_num):
        return nn.Sequential(
            nn.Conv2d(channel_num, kernel_num, 4, 2, 1), 
            nn.GroupNorm(num_groups=32, num_channels=kernel_num),
            nn.ReLU(),
        )
    
    def _deconv(self, channel_num, kernel_num):
        return nn.Sequential(
            nn.ConvTranspose2d(channel_num, kernel_num, 4, 2, 1), 
            nn.GroupNorm(num_groups=32, num_channels=kernel_num),
            nn.ReLU(),
        )

In [ ]:
def reconstruction_loss(x, x_hat):
    return F.mse_loss(x_hat, x, reduction="sum") 

def kl_divergence(mu, logvar):
    return -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp()) 

def vae_loss(x, x_hat, mu, logvar, beta=1.0):
    recon_loss = reconstruction_loss(x, x_hat) / x.size(0)
    kld = kl_divergence(mu, logvar) / x.size(0)

    total_loss = recon_loss + beta * kld
    return total_loss, recon_loss, kld    

### Define optimizer and start training.

In [ ]:
def beta_schedule(epoch, beta, warmup_epochs):
    return beta * min(1.0, epoch / warmup_epochs)


In [ ]:
model = ConvVAE(img_size=64, latent_dim=latent_dim, layers_num=3).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)

train_loader, test_loader = get_data_loaders(dataset=dataset)
class_names = train_loader.dataset.classes
labels = [f"{i}_{name}" for i, name in enumerate(class_names)]
print(labels)
for epoch in range(num_epochs):
    model.train()
    total_loss = 0
    total_recon = 0
    total_kld = 0

    for x, y in tqdm(train_loader, leave=False):
        x = x.to(device)

        x_hat, mu, logvar, z = model(x)

        loss, recon, kld = vae_loss(x, x_hat, mu, logvar, beta=beta_schedule(epoch+1, kl_weight, 40))

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        total_recon += recon.item()
        total_kld += kld.item()

    print(f"Epoch {epoch+1}/{num_epochs} "
          f"Loss={total_loss/len(train_loader):.4f} "
          f"Recon={total_recon/len(train_loader):.4f} "
          f"KLD={total_kld/len(train_loader):.4f}")


In [ ]:
import random
import torch

def encode_images(x):
    model.eval()
    with torch.no_grad():
        mu, _ = model.encode(x.to(device))
    return mu 

def latent_distance(x1, x2):
    return torch.norm(x1 - x2, dim=-1)


In [ ]:
import torch
from collections import defaultdict

# ------------------------
# assumes:
# vae = ConvVAE(...)
# loader = CIFAR100 dataloader
# device = "cuda" or "cpu"
# ------------------------

num_classes = 10
vae = model
vae = vae.to(device)
vae.eval()


In [ ]:


# --------------------------------------------------
# 1) Collect μ (latent mean) grouped by class label
# --------------------------------------------------
def collect_latents_by_class(vae, loader, device):
    latents_by_class = defaultdict(list)

    with torch.no_grad():
        for x, y in loader:
            x = x.to(device)

            # forward pass encoder only
            mu, logvar = vae.encode(x)     # shapes: [B, latent_dim]

            mu = mu.detach().cpu()

            for zi, yi in zip(mu, y):
                latents_by_class[int(yi.item())].append(zi)

    # convert lists to tensors
    for k in latents_by_class:
        latents_by_class[k] = torch.stack(latents_by_class[k])  # [Nk, d]

    return latents_by_class


# --------------------------------------------------
# 2) Centroid–centroid distance matrix
# --------------------------------------------------
def centroid_distance_matrix(latents_by_class, latent_distance):
    C = len(latents_by_class)

    # compute centroids
    centroids = []
    for c in range(C):
        centroids.append(latents_by_class[c].mean(dim=0))
    centroids = torch.stack(centroids)  # [C, d]

    # fill matrix
    mat = torch.zeros((C, C))
    for i in range(C):
        for j in range(C):
            mat[i, j] = latent_distance(centroids[i], centroids[j])

    return mat


# --------------------------------------------------
# 3) True average pairwise class–class distance matrix
# --------------------------------------------------
def avg_l2_between_sets(A, B):
    # A: [N, d], B: [M, d]

    AA = (A * A).sum(dim=1).unsqueeze(1)   # [N,1]
    BB = (B * B).sum(dim=1).unsqueeze(0)   # [1,M]

    AB = A @ B.T                           # [N,M]

    dist_sq = AA + BB - 2 * AB             # [N,M]

    dist = torch.sqrt(torch.clamp(dist_sq, min=1e-8))

    return dist.mean()

def pairwise_average_distance_matrix_l2(latents_by_class):
    C = len(latents_by_class)
    mat = torch.zeros((C, C))

    for i in range(C):
        zi = latents_by_class[i]

        for j in range(i, C):
            zj = latents_by_class[j]

            val = avg_l2_between_sets(zi, zj)

            mat[i, j] = val
            mat[j, i] = val

    return mat


def pairwise_average_distance_matrix(latents_by_class, latent_distance):
    C = len(latents_by_class)
    mat = torch.zeros((C, C))

    for i in range(C):
        zi = latents_by_class[i]      # [Ni, d]

        for j in range(i, C):         # notice: j starts at i
            zj = latents_by_class[j]  # [Nj, d]

            zi_exp = zi.unsqueeze(1)  # [Ni, 1, d]
            zj_exp = zj.unsqueeze(0)  # [1, Nj, d]

            dists = latent_distance(zi_exp, zj_exp)

            val = dists.mean()

            mat[i, j] = val
            mat[j, i] = val           # mirror to bottom-left

    return mat


In [ ]:
latents_by_class = collect_latents_by_class(vae, train_loader, device)
centroid_matrix = centroid_distance_matrix(latents_by_class, latent_distance)
pairwise_matrix = pairwise_average_distance_matrix_l2(latents_by_class)


In [ ]:
import pandas as pd

class_names = train_loader.dataset.classes
labels = [f"{i}_{name}" for i, name in enumerate(class_names)]

def matrix_to_df(matrix, labels):
    """
    matrix: torch tensor [100,100]
    labels: list length 100
    """
    if isinstance(matrix, torch.Tensor):
        matrix = matrix.cpu().numpy()

    df = pd.DataFrame(matrix, index=labels, columns=labels)
    return df


centroid_df = matrix_to_df(centroid_matrix, labels)
pairwise_df = matrix_to_df(pairwise_matrix, labels)

output_path = f"{str(kl_weight).replace('.','')}_{dataset}_latent_distances.xlsx"

with pd.ExcelWriter(output_path, engine="openpyxl") as writer:
    centroid_df.to_excel(writer, sheet_name="centroid_distance")
    pairwise_df.to_excel(writer, sheet_name="pairwise_distance")

print(f"Saved Excel file to: {output_path}")


In [2]:
import torch
import torch.nn.functional as F
import open_clip
from tqdm import tqdm


@torch.no_grad()
def get_clip_latents_by_class(
    train_loader,
    img_size,
    model_name="ViT-B-32",
    pretrained="openai",
    device=None,
    normalize_features=True,
    prompt_template="a photo of a {}",
):
    """
    Returns BOTH:
    (1) image-based class centroids from CLIP image encoder
    (2) text-based class embeddings from CLIP text encoder

    Assumes images are NOT normalized (i.e. in [0,1])

    Returns:
        class_names: list[str]
        image_centroids: Tensor [C, D]
        text_embeddings: Tensor [C, D]
    """
    if device is None:
        device = "cuda" if torch.cuda.is_available() else "cpu"

    dataset = train_loader.dataset
    if not hasattr(dataset, "classes"):
        raise AttributeError("train_loader.dataset must have .classes")

    class_names = list(dataset.classes)
    num_classes = len(class_names)

    # Load CLIP
    model, _, _ = open_clip.create_model_and_transforms(
        model_name=model_name,
        pretrained=pretrained,
    )
    tokenizer = open_clip.get_tokenizer(model_name)
    model = model.to(device).eval()

    # CLIP normalization
    clip_mean = torch.tensor(
        [0.48145466, 0.4578275, 0.40821073], device=device
    ).view(1, 3, 1, 1)
    clip_std = torch.tensor(
        [0.26862954, 0.26130258, 0.27577711], device=device
    ).view(1, 3, 1, 1)

    # ---------- (1) IMAGE CENTROIDS ----------
    feat_sums = None
    counts = torch.zeros(num_classes, device=device)

    for x, y in tqdm(train_loader, desc="CLIP image encoding"):
        x = x.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)

        # resize to CLIP resolution
        visual_size = model.visual.image_size
        if isinstance(visual_size, tuple):
            target_h, target_w = visual_size
        else:
            target_h = target_w = visual_size

        if x.shape[-2:] != (target_h, target_w):
            x = F.interpolate(x, size=(target_h, target_w), mode="bilinear", align_corners=False)

        # CLIP normalization
        x = (x - clip_mean) / clip_std

        feats = model.encode_image(x)

        if normalize_features:
            feats = F.normalize(feats, dim=-1)

        if feat_sums is None:
            D = feats.shape[-1]
            feat_sums = torch.zeros(num_classes, D, device=device)

        feat_sums.index_add_(0, y, feats)
        counts.index_add_(0, y, torch.ones_like(y, dtype=torch.float))

    image_centroids = feat_sums / counts.unsqueeze(1)

    if normalize_features:
        image_centroids = F.normalize(image_centroids, dim=-1)

    # ---------- (2) TEXT EMBEDDINGS ----------
    prompts = [prompt_template.format(name.replace("_", " ")) for name in class_names]
    tokens = tokenizer(prompts).to(device)

    text_embeddings = model.encode_text(tokens)

    if normalize_features:
        text_embeddings = F.normalize(text_embeddings, dim=-1)

    # move to CPU for consistency
    return (
        class_names,
        image_centroids.cpu(),
        text_embeddings.cpu(),
    )

@torch.no_grad()
def get_dinov2_latents_by_class(
    train_loader,
    img_size,
    model_name="dinov2_vitb14",   # dinov2_vits14 / vitb14 / vitl14 / vitg14
    device=None,
    normalize_features=True,
):
    """
    Build per-class image centroids using pretrained DINOv2 image embeddings.

    Assumes:
    - train_loader yields (x, y)
    - x is NOT dataset-normalized, i.e. roughly in [0, 1]
    - train_loader.dataset.classes exists

    Returns:
        class_names: list[str]
        image_centroids: Tensor [C, D]
    """
    if device is None:
        device = "cuda" if torch.cuda.is_available() else "cpu"

    dataset = train_loader.dataset
    if not hasattr(dataset, "classes"):
        raise AttributeError("train_loader.dataset must have .classes")

    class_names = list(dataset.classes)
    num_classes = len(class_names)

    # Official DINOv2 Torch Hub load
    model = torch.hub.load("facebookresearch/dinov2", model_name)
    model = model.to(device).eval()

    # DINOv2 models use ImageNet-style normalization in the official repo
    mean = torch.tensor([0.485, 0.456, 0.406], device=device).view(1, 3, 1, 1)
    std = torch.tensor([0.229, 0.224, 0.225], device=device).view(1, 3, 1, 1)

    feat_sums = None
    counts = torch.zeros(num_classes, device=device)

    for x, y in tqdm(train_loader, desc="DINOv2 image encoding"):
        x = x.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)

        if x.ndim != 4:
            raise ValueError(f"Expected x shape [B, C, H, W], got {tuple(x.shape)}")

        if x.shape[-2] != img_size or x.shape[-1] != img_size:
            raise ValueError(
                f"Expected incoming images to be {img_size}x{img_size}, "
                f"got {x.shape[-2]}x{x.shape[-1]}"
            )

        # DINOv2 ViT-* /14 models are typically used at 224x224 inputs
        if x.shape[-2:] != (224, 224):
            x = F.interpolate(
                x, size=(224, 224), mode="bicubic", align_corners=False
            )

        x = (x - mean) / std

        feats = model(x)   # [B, D]

        if normalize_features:
            feats = F.normalize(feats, dim=-1)

        if feat_sums is None:
            feat_dim = feats.shape[-1]
            feat_sums = torch.zeros(num_classes, feat_dim, device=device)

        feat_sums.index_add_(0, y, feats)
        counts.index_add_(0, y, torch.ones_like(y, dtype=torch.float))

    if (counts == 0).any():
        missing = torch.nonzero(counts == 0).flatten().tolist()
        raise ValueError(f"No samples found for classes: {missing}")

    image_centroids = feat_sums / counts.unsqueeze(1)

    if normalize_features:
        image_centroids = F.normalize(image_centroids, dim=-1)

    return class_names, image_centroids.cpu()

def cosine_distance(x, y):
    return 1 - F.cosine_similarity(x.unsqueeze(0), y.unsqueeze(0)).item()

def matrix_to_df(matrix, labels):
    """
    matrix: torch tensor [100,100]
    labels: list length 100
    """
    if isinstance(matrix, torch.Tensor):
        matrix = matrix.cpu().numpy()

    df = pd.DataFrame(matrix, index=labels, columns=labels)
    return df


ModuleNotFoundError: No module named 'open_clip'

In [15]:
dataset = "tinyimagenet"

train_loader, test_loader = get_data_loaders(dataset=dataset, normalize = False)

In [ ]:
import pandas as pd
def matrix_to_df(matrix, labels):
    """
    matrix: torch tensor [100,100]
    labels: list length 100
    """
    if isinstance(matrix, torch.Tensor):
        matrix = matrix.cpu().numpy()

    df = pd.DataFrame(matrix, index=labels, columns=labels)
    return df
class_names, img_centroids = get_dinov2_latents_by_class(
    train_loader,
    img_size=32,
    model_name="dinov2_vitb14",
    device=device,
)

D_img = 1 - img_centroids @ img_centroids.T
# D_text = 1 - text_embeds @ text_embeds.T

img_df = matrix_to_df(D_img, class_names)
# text_df = matrix_to_df(D_text, class_names)


NameError: name 'get_dinov2_latents_by_class' is not defined

In [16]:
import pandas as pd
import torch

n = 200

# Step 1: random matrix
A = torch.rand(n, n)

# Step 2: make symmetric
A = (A + A.T) / 2

# Step 3: zero out diagonal
A.fill_diagonal_(0)

print(A)

def matrix_to_df(matrix, labels):
    """
    matrix: torch tensor [100,100]
    labels: list length 100
    """
    if isinstance(matrix, torch.Tensor):
        matrix = matrix.cpu().numpy()

    df = pd.DataFrame(matrix, index=labels, columns=labels)
    return df

class_names = list(train_loader.dataset.classes)

img_df = matrix_to_df(A, class_names)
output_path = f"rand_{dataset}_latent_distances.xlsx"

with pd.ExcelWriter(output_path, engine="openpyxl") as writer:
    img_df.to_excel(writer, sheet_name="img_distance")
    # text_df.to_excel(writer, sheet_name="text_distance")

print(f"Saved Excel file to: {output_path}")

tensor([[0.0000, 0.7183, 0.6975,  ..., 0.7452, 0.5258, 0.0569],
        [0.7183, 0.0000, 0.8306,  ..., 0.4893, 0.7087, 0.1433],
        [0.6975, 0.8306, 0.0000,  ..., 0.8870, 0.4949, 0.5414],
        ...,
        [0.7452, 0.4893, 0.8870,  ..., 0.0000, 0.6140, 0.7300],
        [0.5258, 0.7087, 0.4949,  ..., 0.6140, 0.0000, 0.5715],
        [0.0569, 0.1433, 0.5414,  ..., 0.7300, 0.5715, 0.0000]])
Saved Excel file to: rand_tinyimagenet_latent_distances.xlsx


In [ ]:
from torchvision.utils import save_image, make_grid
import matplotlib.pyplot as plt

def show_recon_comparison(x, x_hat, num_samples=8):
    """
    Display original vs reconstructed images side by side horizontally.

    Args:
        x: Original images tensor (batch_size, 1, 28, 28)
        x_hat: Reconstructed images tensor (batch_size, 1, 28, 28)
        num_samples: Number of image pairs to display
    """
    x_display = x[:num_samples]
    x_hat_display = x_hat[:num_samples]

    # Create comparison: first row original, second row reconstructed
    comparison = torch.cat([x_display, x_hat_display], dim=0)

    grid = make_grid(comparison, nrow=num_samples, normalize=True, padding=2)
    plt.figure(figsize=(num_samples * 1.5, 3))
    plt.imshow(grid.permute(1, 2, 0).cpu().numpy(), cmap='gray')
    plt.axis('off')
    plt.title(r'Test reconstruction: $x$ (1st row) vs $\hat{x}$ (2nd row)' + f' (KL weight = {kl_weight})')
    plt.show()

if True:
    train_loader, test_loader = get_data_loaders(dataset=dataset)
    # Evaluate Convolutional VAE and generate images
    model.eval()

    print("Generating images from VAE...")
    total_test_recon_loss = 0

    # Test reconstruction
    with torch.no_grad():
        for batch_idx, (x, _) in enumerate(test_loader):
            x = x.to(device)
            x_hat, _, _, _ = model(x)
            if batch_idx == 0:
                show_recon_comparison(x, x_hat, num_samples=16)
            recon_loss = reconstruction_loss(x, x_hat)
            total_test_recon_loss += recon_loss.item()

    print(f"Total test reconstruction loss: {total_test_recon_loss}")

In [ ]:
def sample_prior(n_samples=1):
        """
        Sample from the prior distribution.
        """
        z = torch.randn(n_samples, latent_dim).to(device=device)
        return z

if True:
    model.eval()

    # Generate new images by sampling from prior
    with torch.no_grad():
        num_samples = 64
        z = sample_prior(n_samples=num_samples)
        generated_images = model.decode(z)

        # Display in
        grid = make_grid(generated_images, nrow=8, normalize=True, padding=2)
        plt.figure(figsize=(5, 5))
        plt.imshow(grid.permute(1, 2, 0).cpu().numpy())
        plt.axis('off')
        plt.title(f'Generated Images from VAE w/ KL weight = {kl_weight}')
        plt.show()
